# BinWaves example in Cantabria (Validation)

**In this notebook**: 
<br><br>
Here waves are reconstructed at the buoy location for comparison.
<br><br>
Steps:
- Buoy is loaded.
- Kp propagation coefficients and hindcast reconstruction is made at the buoy location.
- Comparison plots and statistics are shown.

In [1]:
import pandas as pd
import xarray as xr
import numpy as np

# Define buoy locations dictionary with both UTM and WGS84 coordinates
buoy_locations_dict = {
    '44088': {
        'utm': [514397.61, 4051843.74],  # UTM coordinates
        'wgs84': [36.6120, -74.8390]     # WGS84 coordinates (lat, lon)
    },
    '44100': {
        'utm': [446728.66, 4012728.08],  # UTM coordinates
        'wgs84': [36.2580, -75.5930]     # WGS84 coordinates (lat, lon)
    },
    '44086': {
        'utm': [462056.64, 3984141.31],  # UTM coordinates
        'wgs84': [36.0010, -75.4210]     # WGS84 coordinates (lat, lon)
    },
    '44056': {
        'utm': [435811.25, 4006367.97],  # UTM coordinates
        'wgs84': [36.2000, -75.7140]     # WGS84 coordinates (lat, lon)
    },
    '44095': {
        'utm': [470164.24, 3956270.58],  # UTM coordinates
        'wgs84': [35.7500, -75.3300]     # WGS84 coordinates (lat, lon)
    },
    '41120': {
        'utm': [474075.136, 3901692.114], # UTM coordinates
        'wgs84': [35.2580, -75.2850]     # WGS84 coordinates (lat, lon)
    },
    '41025': {
        'utm': [458576.64, 3874246.18],  # UTM coordinates
        'wgs84': [35.0100, -75.4540]     # WGS84 coordinates (lat, lon)
    }
}

def find_site_index(kp_coeffs, target_x, target_y, tolerance=1.0):
    """
    Find the site index in kp_coeffs that matches the target coordinates.
    
    Parameters:
    -----------
    kp_coeffs : xarray.Dataset
        The kp coefficients dataset
    target_x : float
        Target UTM x coordinate
    target_y : float
        Target UTM y coordinate
    tolerance : float
        Tolerance for coordinate matching (in meters)
        
    Returns:
    --------
    int
        The site index that matches the coordinates
    """
    # Get all site coordinates
    site_x = kp_coeffs.utm_x.values
    site_y = kp_coeffs.utm_y.values
    
    # Calculate distances to all sites
    distances = np.sqrt((site_x - target_x)**2 + (site_y - target_y)**2)
    
    # Find the site with minimum distance
    site_index = np.argmin(distances)
    
    # Check if the closest site is within tolerance
    if distances[site_index] > tolerance:
        print(f"Warning: Closest site is {distances[site_index]:.2f} meters away")
    
    return site_index

def load_buoy_data(buoy_id, year=None):
    """
    Load buoy data and return both the wave data and location coordinates.
    
    Parameters:
    -----------
    buoy_id : str
        The buoy ID (e.g., '44100')
    year : str, optional
        The year to filter data for (default: None)
        
    Returns:
    --------
    tuple
        (buoy_waves, buoy_location, kp_coeffs)
    """
    # Load buoy data
    buoy_waves = pd.read_pickle(f"common_inputs/buoy_{buoy_id}_bulk_parameters.pkl").sort_index()
    if year:
        buoy_waves = buoy_waves.loc[year]
    buoy_waves = buoy_waves.dropna(subset=['Hs_Buoy', 'Tp_Buoy', 'Dir_Buoy'])
    
    # Get buoy location (both UTM and WGS84)
    buoy_location = buoy_locations_dict[buoy_id]
    
    # Load kp coefficients
    kp_coeffs = xr.open_dataset("outputs_NC_SC/kp_coefficients.nc")
    
    # Find the correct site index
    site_index = find_site_index(kp_coeffs, buoy_location['utm'][0], buoy_location['utm'][1])
    
    # Select the correct site
    kp_coeffs = kp_coeffs.isel(site=[site_index])
    
    return buoy_waves, buoy_location, kp_coeffs

# Example usage:
buoy_id = "44056"  # Change this to use different buoys
buoy_waves, buoy_location, kp_coeffs = load_buoy_data(buoy_id)

# Print the coordinates for verification
print(f"Buoy {buoy_id} coordinates:")
print(f"UTM: {buoy_location['utm']}")
print(f"WGS84: {buoy_location['wgs84']}")
print(f"\nSelected kp_coeffs site coordinates:")
print(f"UTM: {kp_coeffs.utm_x.values}, {kp_coeffs.utm_y.values}")

Buoy 44056 coordinates:
UTM: [435811.25, 4006367.97]
WGS84: [36.2, -75.714]

Selected kp_coeffs site coordinates:
UTM: [435811.25], [4006367.97]


In [ ]:
from utils.operations import transform_ERA5_spectrum
import numpy as np
model_parameters = pd.read_csv("NC_SC/swan_cases.csv").to_dict(orient="list")

# Load interest spectra
# offshore_spectra, offshore_spectra_case = (  # Unpack both values from the tuple
#     transform_ERA5_spectrum(
#         era5_spectrum=xr.open_dataset("common_inputs/satellite_corrected_44088_spectrum_cal.nc"),
#         subset_parameters=model_parameters,
#         available_case_num=kp_coeffs.case_num.values,
#     )
# )

offshore_spectra, offshore_spectra_case = transform_ERA5_spectrum(
    era5_spectrum=xr.open_dataset("common_inputs/satellite_corrected_44088_spectrum_cal.nc"),
    subset_parameters=model_parameters,
    available_case_num=kp_coeffs.case_num.values,
    satellite_correction=True,
    target_epsg="EPSG:32618" # This is the UTM zone 18N for North Carolina
)


offshore_spectra_case.coords['longitude'] = np.float32(buoy_location['wgs84'][1]+360)
offshore_spectra_case.coords['latitude'] = np.float32(buoy_location['wgs84'][0])

# Add attributes to the coordinates
offshore_spectra_case.coords['longitude'].attrs = {
    'units': 'degrees_east',
    'long_name': 'Longitude'
}
offshore_spectra_case.coords['latitude'].attrs = {
    'units': 'degrees_north',
    'long_name': 'Latitude'
}
offshore_spectra_case

/home/grupos/geocean/montanoj/miniconda3/envs/calval/lib/python3.13/site-packages/xarray/core/accessor_dt.py:163: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  field_values = method(freq=freq).values


In [3]:
# import xarray as xr

# # Load the dataset
# offshore_spectra_case = xr.open_dataset("outputs/jen_north_carolina_spec_utm_unique.nc")


# import numpy as np
# times = offshore_spectra_case.time.to_index()
# unique_times, unique_idx = np.unique(offshore_spectra_case.time.values, return_index=True)
# offshore_spectra_case = offshore_spectra_case.isel(time=unique_idx)
# offshore_spectra_case.to_netcdf("outputs/jen_north_carolina_spec_utm_unique.nc")

In [2]:
# from bluemath_tk.waves.binwaves import reconstruc_spectra
# # Reconstruct spectra

# reconstructed_onshore_spectra = reconstruc_spectra(
#     offshore_spectra=offshore_spectra_case.sel(time=buoy_waves.index, method="nearest"),
#     kp_coeffs=kp_coeffs,
# )
# reconstructed_onshore_spectra
from bluemath_tk.waves.binwaves import reconstruc_spectra
from utils.operations import remove_duplicate_times


# For offshore_spectra_case (assuming it's an xarray Dataset)
_, index = np.unique(offshore_spectra_case.time.values, return_index=True)
offshore_spectra_case = offshore_spectra_case.isel(time=index)

# For buoy_waves (assuming it's a pandas DataFrame)
buoy_waves = buoy_waves[~buoy_waves.index.duplicated(keep='first')]

# Now reconstruct spectra
reconstructed_onshore_spectra = reconstruc_spectra(
    offshore_spectra=offshore_spectra_case.sel(time=buoy_waves.index, method="nearest"),
    kp_coeffs=kp_coeffs,
)

NameError: name 'offshore_spectra_case' is not defined

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(20, 10), sharex=True)

for i, var in enumerate(['Hs_Buoy', 'Tp_Buoy', 'Dir_Buoy']):
    axes[i].plot(buoy_waves.index, buoy_waves[var], label=var)
    # Overlay NaN locations
    nan_locs = buoy_waves[var].isna()
    axes[i].scatter(buoy_waves.index[nan_locs], [axes[i].get_ylim()[0]]*nan_locs.sum(), 
                    color='red', marker='|', s=100, label='NaN')
    axes[i].set_ylabel(var)
    axes[i].legend()

axes[-1].set_xlabel('Time')
plt.tight_layout()
plt.show()

In [ ]:
# from utils.plotting import plot_wave_series
import importlib
import utils.plotting as plotting
importlib.reload(plotting)
from utils.plotting import plot_wave_series  # or whatever function you need


# First ensure unique timestamps in offshore_spectra
_, unique_idx = np.unique(offshore_spectra.time.values, return_index=True)
offshore_spectra_unique = offshore_spectra.isel(time=unique_idx)

# Now plot with the unique timestamps
# plot_wave_series(
#     buoy_data=buoy_waves,
#     binwaves_data=reconstructed_onshore_spectra.rename({"kps": "efth"})
#         .squeeze()
#         .drop_duplicates('time')
#         .sel(time=buoy_waves.index, method="nearest")
#         .spec,
#     offshore_data=offshore_spectra_unique.sel(time=buoy_waves.index, method="nearest").spec,
#     times=buoy_waves.index.values,
#     buoyId=buoy_id,
#     save_dir='outputs_NC_SC/Figures/'  # Specify where to save the figures
# )

plot_wave_series(
    buoy_data=buoy_waves,
    binwaves_data=reconstructed_onshore_spectra.rename({"kps": "efth"})
        .squeeze()
        .drop_duplicates('time')
        .sel(time=buoy_waves.index, method="nearest")
        .spec,
    offshore_data=offshore_spectra_unique.sel(time=buoy_waves.index, method="nearest").spec,
    times=buoy_waves.index.values,
    buoyId=buoy_id,
    save_dir='outputs_NC_SC/Figures/'
)

In [ ]:
from utils.data_export import save_wave_series_to_csv
save_wave_series_to_csv(
    buoy_data=buoy_waves,
    binwaves_data=reconstructed_onshore_spectra.rename({"kps": "efth"})
        .squeeze()
        .drop_duplicates('time')  # Add this line to remove duplicates
        .sel(time=buoy_waves.index, method="nearest")
        .spec,
    times=buoy_waves.index.values,
    output_file=f'outputs_NC_SC/buoy_{buoy_id}_validation_sat.csv'
)
